# Example script for Hackathon

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have limited chances to make queries.

In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

import os
import esm
from tqdm.auto import tqdm

## 1. Collect Training Data

Upload `sequence.fasta`, `train.csv`, and `test.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [2]:
with open('Hackathon_data/Hackathon_data/sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [3]:
len(sequence_wt)

656

In [4]:

def get_mutated_sequence(mut, sequence_wt):
    '''
    Adds the specified mutation into the wild-type sequence.

    Params: 
        mut (str): The mutation to be applied.
        sequence_wt (str): The wild-type sequence to which the mutation will be applied.
    Returns:
        A deep copy of the mutated sequence string.
    '''
    # wt - wild type; pos - position; mt - mutation
    wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

    sequence = deepcopy(sequence_wt)

    return sequence[:pos] + mt + sequence[pos+1:]

In [5]:
df_train = pd.read_csv('Hackathon_data/Hackathon_data/train.csv')
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_train

,mutant,DMS_score,sequence
0,M0Y,0.2730,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.2857,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.2153,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.3122,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.2180,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...,...
1135,P347D,0.3876,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1136,P347C,0.1837,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1137,P347A,0.4611,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1138,P347M,0.2412,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [6]:
df_test = pd.read_csv('Hackathon_data/Hackathon_data/test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [7]:
# TODO: integrate the query data that you acquired each round into df_train

## 2. Train a Prediction Model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

### Hyperparameters

In [8]:
seq_length = 656
seed = 0 # seed for splitting the validation set
val_ratio = 0.2 # proportion of validation set

### ProteinDataset
Convert amino acids to machine-readable data via embedding space used by ESM-2.

In [ ]:
def gen_emb_from_df(df, out_dir="esm_embeddings_variants", device="cuda:0", batch_size=8):
    '''
    Method to generate embeddings and store them in target output directory.
    '''
    os.makedirs(out_dir, exist_ok=True)

    if isinstance(device, torch.device):
        use_cuda = device.type == "cuda"
    else:
        use_cuda = str(device).startswith("cuda")

    # Each item is (name_for_file, sequence)
    data = [(m, s[:1000]) for m, s in df[["mutant", "sequence"]].drop_duplicates().values]
    print(f"Number of unique variants: {len(data)}")

    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    model.to(device).eval()
    if use_cuda:
        model.half()  # Half-prec. to shorten runtime on GPU

    for i in tqdm(range(int(np.ceil(len(data) / batch_size)))):
        batch = data[i * batch_size:(i + 1) * batch_size]

        # Cache skip
        if all(os.path.exists(os.path.join(out_dir, f"{name}.pt")) for name, _ in batch):
            continue

        labels, strs, tokens = batch_converter(batch)
        lens = (tokens != alphabet.padding_idx).sum(1)
        tokens = tokens.to(device)

        with torch.no_grad():
            if use_cuda:
                with torch.autocast(device_type="cuda"):
                    out = model(tokens, repr_layers=[33])
            else:
                out = model(tokens, repr_layers=[33])

        reps = out["representations"][33].detach().cpu()

        for k, L in enumerate(lens):
            name = batch[k][0]
            path = os.path.join(out_dir, f"{name}.pt")
            if os.path.exists(path):
                continue

            # Exclude BOS/EOS tokens for cleaner mean pooling
            seq_tokens = reps[k, 1:L-1]
            seq_mean = seq_tokens.mean(0)
            torch.save(seq_mean, path)

class DmsESMDataset(Dataset):
    def __init__(self, df, emb_dir, is_train=True):
        self.df = df.reset_index(drop=True)
        self.emb_dir = emb_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        mutant = self.df.loc[idx, "mutant"]
        emb = torch.load(os.path.join(self.emb_dir, f"{mutant}.pt")).float()
        if self.is_train:
            y = torch.tensor(self.df.loc[idx, "DMS_score"], dtype=torch.float32)
            return emb, y
        return emb, torch.tensor(0.0, dtype=torch.float32)

In [ ]:
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No CUDA device found; using CPU.")

# Run on GPU 0 when available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

True
NVIDIA GeForce RTX 5070 Laptop GPU


In [23]:
# Combine our datasets for embedding, them split them into train / test sets afterward. Each has mutated sequence variants stored as a separate .pt embedding.
all_df = pd.concat(
    [df_train[["mutant", "sequence"]], df_test[["mutant", "sequence"]]],
    ignore_index=True
).drop_duplicates("mutant")

gen_emb_from_df(all_df, out_dir="esm_embeddings_variants", device=device, batch_size=8)

Number of unique variants: 12464


  0%|          | 0/1558 [00:00<?, ?it/s]

In [20]:
# gen_emb('Hackathon_Data/Hackathon_data/sequence.fasta', out_dir='esm_embeddings_train')
# gen_emb('Hackathon_Data/Hackathon_data/', out_dir='esm_embeddings_test')

Number of sequences: 1


  0%|          | 0/1 [00:00<?, ?it/s]

In [56]:
# Use simple MLP model to predict from ESM-2 embeddings.
class MLPClassifier(nn.Module):
    def __init__(self, input_dim=1280, hidden_dim=640, dropout_p=0.1):
        super(MLPClassifier, self).__init__()
        
        # Only need to predict a single fitness score every time.
        output_dim = 1

        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid() # Ensures fitness scores in range (0,1).
        )

    def forward(self, x):
        return self.layers(x)

In [57]:
def train_model_esm(model, train_dataset, val_dataset, epochs=100, batch_size=256, lr=1e-3, patience=10, device='cuda:0'):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    # Use MSE loss to handle bounded regression.
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    best_val_spearman = -np.inf
    best_ckpt = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []
        train_preds = []
        train_targets = []

        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device).float()

            optimizer.zero_grad()
            outputs = model(inputs).squeeze(-1)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.append(outputs.detach().cpu())
            train_targets.append(targets.detach().cpu())

        train_preds = torch.cat(train_preds).numpy()
        train_targets = torch.cat(train_targets).numpy()
        train_spearman = spearmanr(train_preds, train_targets).statistic

        # Validation
        model.eval()
        val_losses = []
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                targets = targets.to(device).float()
                outputs = model(inputs).squeeze(-1)

                val_losses.append(criterion(outputs, targets).item())
                val_preds.append(outputs.detach().cpu())
                val_targets.append(targets.detach().cpu())

        val_preds = torch.cat(val_preds).numpy()
        val_targets = torch.cat(val_targets).numpy()
        val_spearman = spearmanr(val_preds, val_targets).statistic
        mean_train_loss = float(np.mean(train_losses))
        mean_val_loss = float(np.mean(val_losses))

        if np.isnan(train_spearman):
            train_spearman = 0.0
        if np.isnan(val_spearman):
            val_spearman = -1.0

        print(
            f"Epoch {epoch+1}: "
            f"Train Loss={mean_train_loss:.4f}, Train Spearman={train_spearman:.4f}, "
            f"Val Loss={mean_val_loss:.4f}, Val Spearman={val_spearman:.4f}"
        )

        # Early stopping on validation Spearman (ranking quality)
        if val_spearman > best_val_spearman:
            best_val_spearman = val_spearman
            best_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if best_ckpt is None:
        best_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return model, best_ckpt

In [58]:
# Split data into train and validation sets, then form DmsESMDataset() classes from each.
df_train_split, df_val_split = train_test_split(
    df_train, test_size=val_ratio, random_state=seed, shuffle=True
)

train_ds = DmsESMDataset(df_train_split, "esm_embeddings_variants", is_train=True)
val_ds = DmsESMDataset(df_val_split, "esm_embeddings_variants", is_train=True)
test_ds = DmsESMDataset(df_test, "esm_embeddings_variants", is_train=False)

test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [61]:
# --------------- Train our model ---------------
model_esm = MLPClassifier(input_dim=1280,
                          hidden_dim=640, 
                          dropout_p=0.3).to(device)
# TODO Currently using test values to make sure this works.
model_esm, best_ckpt_esm = train_model_esm(
    model_esm,
    train_ds,
    val_ds,
    epochs=10,  # 500
    batch_size=256,
    lr=1e-3,  # 1e-4
    patience=5,  # 50
    device=device,
 )
model_esm.load_state_dict(best_ckpt_esm)


# --------------- Test our model ---------------
model_esm.eval()
preds = []
with torch.no_grad():
    for sequences, _ in test_loader:
        # Inference on the test set
        sequences = sequences.to(device)
        outputs = model_esm(sequences)
        dms_score_pred = outputs.squeeze(-1).cpu().numpy()
        preds.extend(dms_score_pred.tolist())

df_preds = pd.DataFrame(preds)
df_preds.to_csv('test_predictions.csv', index=False, header=False)

Epoch 1: Train Loss=0.1142, Train Spearman=0.1192, Val Loss=0.1087, Val Spearman=-0.0914
Epoch 2: Train Loss=0.0422, Train Spearman=0.3422, Val Loss=0.1000, Val Spearman=0.2328
Epoch 3: Train Loss=0.0386, Train Spearman=0.3394, Val Loss=0.0919, Val Spearman=0.2132
Epoch 4: Train Loss=0.0383, Train Spearman=0.4369, Val Loss=0.0645, Val Spearman=0.1756
Epoch 5: Train Loss=0.0354, Train Spearman=0.4954, Val Loss=0.0514, Val Spearman=0.2386
Epoch 6: Train Loss=0.0328, Train Spearman=0.4852, Val Loss=0.0493, Val Spearman=0.2838
Epoch 7: Train Loss=0.0285, Train Spearman=0.5227, Val Loss=0.0501, Val Spearman=0.2772
Epoch 8: Train Loss=0.0279, Train Spearman=0.5460, Val Loss=0.0497, Val Spearman=0.3098
Epoch 9: Train Loss=0.0250, Train Spearman=0.5845, Val Loss=0.0494, Val Spearman=0.3032
Epoch 10: Train Loss=0.0226, Train Spearman=0.6193, Val Loss=0.0500, Val Spearman=0.3093


In [66]:
# Show current ESM-based predictions.
df_test[['mutant', 'DMS_score_predicted']].head(n=10)

,mutant,DMS_score_predicted
0,V1D,0.299826
1,V1Y,0.303264
2,V1C,0.302393
3,V1A,0.305476
4,V1E,0.302815
5,V1W,0.301892
6,V1T,0.301895
7,V1R,0.300785
8,V1Q,0.303569
9,V1S,0.303379


In [67]:
# Save predictions to .csv.
df_test[['mutant', 'DMS_score_predicted']].to_csv('test_predictions.csv', index=False)

## 3. Select Query for Next Round

In [68]:
df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

,mutant,sequence,DMS_score_predicted
280,E15N,MVNEARGNSSLNPCLNGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.310153
88,R5N,MVNEANGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.308870
4661,D281Q,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.308391
8994,L533Q,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.308002
7160,L436K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.307960
...,...,...,...
4109,T243Q,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.304812
3629,K211A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.304787
3909,E231Q,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.304765
77,R5A,MVNEAAGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.304752


In [69]:
# Example: randomly select 100 test variants to be queried.
# Note: Random selection may not be a good strategy
# TODO: Select query mutants for the next round based on your own criteria

queries = np.random.choice(df_test.mutant.values, size=100, replace=False)
queries

array(['T646I', 'D80A', 'E469V', 'D360N', 'E53N', 'K596M', 'I562V',
       'E457P', 'H447N', 'Q484W', 'W164C', 'G71K', 'E432M', 'G440F',
       'D494K', 'R43N', 'T106G', 'V433M', 'I487M', 'A395R', 'L459H',
       'L589P', 'R534Y', 'D401L', 'D624Q', 'S391K', 'R593S', 'S218Q',
       'I286N', 'T606Y', 'P485A', 'L533H', 'K529L', 'D37S', 'L264P',
       'H95N', 'W366L', 'D27N', 'E202F', 'K47D', 'L404A', 'F59C', 'G186S',
       'E513L', 'S218N', 'L597H', 'R444K', 'R45A', 'D622C', 'K400T',
       'F171I', 'V433I', 'F587Q', 'P34L', 'F514Q', 'L470M', 'N74T', 'E3C',
       'E432S', 'D288N', 'A72I', 'W582G', 'R356H', 'I374N', 'F79N',
       'E122V', 'P12E', 'T480N', 'N455S', 'E469I', 'A352H', 'E552N',
       'D209T', 'R78F', 'K179A', 'F365M', 'G498H', 'W388F', 'K448W',
       'Q163L', 'Q476W', 'S411M', 'I478W', 'A649W', 'Q508L', 'G85P',
       'E167H', 'L465G', 'F405N', 'K530Y', 'T248Y', 'R344L', 'I562Y',
       'I611F', 'H140C', 'T128P', 'N482Q', 'V361G', 'W590N', 'E46Y'],
      dtype=object)

In [70]:
with open('query.txt', 'w') as f:
  for mutant in queries:
    f.write(mutant+'\n')